## User-defined bridges & DNS

Creating a **user-defined bridge** is one command, and it's the network you'll actually use:

```bash
docker network create app-net
```

Versus the default bridge, it gives you three things that matter: **embedded DNS** (resolve other containers by their `--name`), **isolation** (only containers you attach can talk), and easy **runtime connect/disconnect**. The standard pattern is one bridge per application unit — a service plus its DB and cache — with everything attached to it:

```bash
docker network create app-net
docker run -d --name db  --network app-net postgres:16
docker run -d --name api --network app-net --env DB_HOST=db myorg/api
```

Inside `api`, the hostname **`db` resolves to the db container's IP** — no `-p` between them, because same-bridge containers already talk on their container ports. This is *the* everyday networking move: name-based service discovery for free.

**How the DNS works.** On a user-defined network, the container's `/etc/resolv.conf` points at Docker's **embedded resolver** at `127.0.0.11`, which:

- resolves other container **names** (and `--network-alias`es) on the same network to their IPs, and
- forwards everything else to the host's upstream DNS.

A few knobs when you need them:

- **`--network-alias web,frontend`** — extra names this container answers to; *several* containers can share one alias to round-robin between them.
- **`--add-host db.internal:10.0.0.5`** — a static `/etc/hosts` entry (a fixed name→IP with no DNS).
- **`--dns 1.1.1.1`** — override the upstream resolver.

The rule of thumb: **one user-defined bridge per app; let containers find each other by `--name`.** That single habit replaces almost all manual IP wrangling.